<a href="https://colab.research.google.com/github/NikhilGeorge01/ML-Prac/blob/main/MLAssignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from collections import deque

In [1]:
import gym
import numpy as np
import random
import math
import matplotlib.pyplot as plt


# Compatibility patch for NumPy and Gym
if not hasattr(np, "bool8"):
  np.bool8 = np.bool_


# Use updated CartPole environment
env = gym.make('CartPole-v1', render_mode='human')

# Keras imports with fallback
try:
    from keras.layers import Dense, Input, Lambda
    from keras.models import Sequential, Model
    from keras.optimizers import Adam
    import keras.backend as K
    from keras.utils import plot_model
except Exception:
    # Fallback to tensorflow.keras
    from tensorflow.keras.layers import Dense, Input, Lambda
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.optimizers import Adam
    import tensorflow.keras.backend as K
    try:
        from tensorflow.keras.utils import plot_model
    except Exception:
        plot_model = None

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-pack

In [2]:
def visualize_cartpole(env_name="CartPole-v0", episodes=100):
    env = gym.make(env_name)
    for i_episode in range(episodes):
        obs = env.reset()
        done = False
        t = 0
        while not done:
            env.render()
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            t += 1
            if done:
                print(f"Episode {i_episode} done after {t} steps")
                break
    env.close()


In [3]:
# Helper to bucketize continuous state values
def bucketize_state_value(state_value, state_value_bounds, no_buckets):
    bucket_indices = []
    for i in range(len(state_value)):
        if state_value[i] <= state_value_bounds[i][0]:
            bucket_index = 0
        elif state_value[i] >= state_value_bounds[i][1]:
            bucket_index = no_buckets[i] - 1
        else:
            bound_width = state_value_bounds[i][1] - state_value_bounds[i][0]
            offset = (no_buckets[i] - 1) * state_value_bounds[i][0] / bound_width
            scaling = (no_buckets[i] - 1) / bound_width
            bucket_index = int(round(scaling * state_value[i] - offset))
        bucket_indices.append(bucket_index)
    return tuple(bucket_indices)


def q_learning_cartpole(
    env_name='CartPole-v0',
    no_buckets=(1, 1, 6, 3),
    max_episodes=1000,
    max_time_steps=250,
    min_explore_rate=0.1,
    min_learning_rate=0.1,
    discount=0.99,
    solved_time=199,
    streak_to_end=120,
):
    env = gym.make(env_name)
    no_actions = env.action_space.n

    # state bounds (fix infinities)
    state_value_bounds = list(zip(env.observation_space.low, env.observation_space.high))
    # Clip unreasonable bounds for cart-pole
    state_value_bounds = list(state_value_bounds)
    state_value_bounds[1] = (-0.5, 0.5)
    state_value_bounds[3] = (-math.radians(50), math.radians(50))

    # define q_value_table - it has a dimension of 1 x 1 x 6 x 3 x 2
    q_value_table = np.zeros(no_buckets + (no_actions,))

    def select_explore_rate(x):
        return max(min_explore_rate, min(1.0, 1.0 - math.log10((x + 1) / 25)))

    def select_learning_rate(x):
        return max(min_learning_rate, min(1.0, 1.0 - math.log10((x + 1) / 25)))

    def select_action(state_value, explore_rate):
        if random.random() < explore_rate:
            return env.action_space.sample()  # explore
        else:
            return int(np.argmax(q_value_table[state_value]))

    no_streaks = 0
    for episode_no in range(max_episodes):
        explore_rate = select_explore_rate(episode_no)
        learning_rate = select_learning_rate(episode_no)

        observation = env.reset()
        previous_state_value = bucketize_state_value(observation, state_value_bounds, no_buckets)
        done = False
        time_step = 0
        while not done and time_step < max_time_steps:
            action = select_action(previous_state_value, explore_rate)
            observation, reward_gain, done, info = env.step(action)
            state_value = bucketize_state_value(observation, state_value_bounds, no_buckets)

            # update q_value_table
            best_q_value = np.max(q_value_table[state_value])
            old_q = q_value_table[previous_state_value][action]
            q_value_table[previous_state_value][action] += learning_rate * (
                reward_gain + discount * best_q_value - old_q
            )

            previous_state_value = state_value
            time_step += 1

        if time_step >= solved_time:
            no_streaks += 1
        else:
            no_streaks = 0

        if no_streaks > streak_to_end:
            print(f'CartPole problem is solved after {episode_no} episodes.')
            break

    env.close()
    return q_value_table

In [4]:
class DQNAgent:
    def __init__(
        self,
        state_size,
        action_size,
        ddqn_flag=False,
        dueling_option=None,  # None, 'avg', or 'max' or 'naive'
        learning_rate=0.001,
        discount_factor=0.99,
    ):
        self.state_size = state_size
        self.action_size = action_size
        # hyper parameters for DQN
        self.discount_factor = discount_factor
        self.learning_rate = learning_rate
        self.epsilon = 1.0  # explore rate
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.batch_size = 24
        self.train_start = 1000
        self.dueling_option = dueling_option
        self.ddqn = ddqn_flag
        # create replay memory using deque
        self.memory = deque(maxlen=2000)
        # create main model and target model
        self.model = self.build_model()
        self.target_model = self.build_model()
        # initialize target model
        self.target_model.set_weights(self.model.get_weights())

    def build_model(self):
        # If dueling_option is set, build dueling network
        if self.dueling_option in ('avg', 'max', 'naive'):
            network_input = Input(shape=(self.state_size,), name='network_input')
            A1 = Dense(24, activation='relu', name='A1')(network_input)
            A2 = Dense(24, activation='relu', name='A2')(A1)
            A3 = Dense(self.action_size, activation='linear', name='A3')(A2)
            V3 = Dense(1, activation='linear', name='V3')(A2)
            if self.dueling_option == 'avg':
                network_output = Lambda(lambda x: x[1] + (x[0] - K.mean(x[0], axis=1, keepdims=True)),
                                        output_shape=(self.action_size,))([A3, V3])
            elif self.dueling_option == 'max':
                network_output = Lambda(lambda x: x[1] + (x[0] - K.max(x[0], axis=1, keepdims=True)),
                                        output_shape=(self.action_size,))([A3, V3])
            elif self.dueling_option == 'naive':
                network_output = Lambda(lambda x: x[0] + x[1], output_shape=(self.action_size,))([A3, V3])
            model = Model(network_input, network_output)
            model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
            return model
        # Standard DQN
        model = Sequential()
        model.add(Dense(24, input_dim=self.state_size, activation='relu'))
        model.add(Dense(24, activation='relu'))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model

    def update_target_model(self, tau=None):
        # If tau provided, do Polyak averaging
        if tau is not None:
            weights = self.model.get_weights()
            target_weights = self.target_model.get_weights()
            for i in range(len(target_weights)):
                target_weights[i] = weights[i] * tau + target_weights[i] * (1 - tau)
            self.target_model.set_weights(target_weights)
        else:
            self.target_model.set_weights(self.model.get_weights())

    def select_action(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        q_value = self.model.predict(state)
        return int(np.argmax(q_value[0]))

    def add_experience(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def get_target_q_value(self, next_state, reward):
        # returns max Q value among next state's actions (for target calculation)
        if self.ddqn:
            action = int(np.argmax(self.model.predict(next_state)[0]))
            max_q_value = self.target_model.predict(next_state)[0][action]
        else:
            max_q_value = np.amax(self.target_model.predict(next_state)[0])
        return max_q_value

    def experience_replay(self):
        if len(self.memory) < self.train_start:
            return
        batch_size = min(self.batch_size, len(self.memory))
        mini_batch = random.sample(self.memory, batch_size)
        state_batch, q_values_batch = [], []
        for state, action, reward, next_state, done in mini_batch:
            q_values_cs = self.model.predict(state)
            max_q_value_ns = self.get_target_q_value(next_state, reward)
            if done:
                q_values_cs[0][action] = reward
            else:
                q_values_cs[0][action] = reward + self.discount_factor * max_q_value_ns
            state_batch.append(state[0])
            q_values_batch.append(q_values_cs[0])
        self.model.fit(np.array(state_batch), np.array(q_values_batch), batch_size=batch_size, epochs=1, verbose=0)
        # decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

In [5]:
class SumTree(object):
    def __init__(self, capacity):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1)
        self.data = np.zeros(capacity, dtype=object)
        self.data_pointer = 0

    def add(self, priority, data):
        tree_index = self.data_pointer + self.capacity - 1
        self.data[self.data_pointer] = data
        self.update(tree_index, priority)
        self.data_pointer += 1
        if self.data_pointer >= self.capacity:
            self.data_pointer = 0

    def update(self, tree_index, priority):
        change = priority - self.tree[tree_index]
        self.tree[tree_index] = priority
        while tree_index != 0:
            tree_index = (tree_index - 1) // 2
            self.tree[tree_index] += change

    def get_leaf(self, v):
        parent_index = 0
        while True:
            left = 2 * parent_index + 1
            right = left + 1
            if left >= len(self.tree):
                leaf_index = parent_index
                break
            else:
                if v <= self.tree[left]:
                    parent_index = left
                else:
                    v -= self.tree[left]
                    parent_index = right
        data_index = leaf_index - self.capacity + 1
        return leaf_index, self.tree[leaf_index], self.data[data_index]

    @property
    def total_priority(self):
        return self.tree[0]


class Memory(object):
    PER_e = 0.01
    PER_a = 0.6
    PER_b = 0.4
    PER_b_increment_per_sampling = 0.001
    absolute_error_upper = 1.0

    def __init__(self, capacity):
        self.tree = SumTree(capacity)

    def store(self, experience):
        max_priority = np.max(self.tree.tree[-self.tree.capacity:])
        if max_priority == 0:
            max_priority = self.absolute_error_upper
        self.tree.add(max_priority, experience)

    def sample(self, n):
        minibatch = []
        b_idx = np.empty((n,), dtype=np.int32)
        priority_segment = self.tree.total_priority / n
        for i in range(n):
            a, b = priority_segment * i, priority_segment * (i + 1)
            value = np.random.uniform(a, b)
            index, priority, data = self.tree.get_leaf(value)
            b_idx[i] = index
            minibatch.append([data[0], data[1], data[2], data[3], data[4]])
        return b_idx, minibatch

    def batch_update(self, tree_idx, abs_errors):
        abs_errors += self.PER_e
        clipped_errors = np.minimum(abs_errors, self.absolute_error_upper)
        ps = np.power(clipped_errors, self.PER_a)
        for ti, p in zip(tree_idx, ps):
            self.tree.update(ti, p)

In [6]:
def run_dqn_cartpole(
    env_name='CartPole-v0',
    episodes=500,
    ddqn_flag=False,
    dueling_option=None,
    memory_size=2000,
):
    env = gym.make(env_name)
    env.seed(0)
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    agent = DQNAgent(state_size, action_size, ddqn_flag, dueling_option)
    score = []
    for e in range(episodes):
        done = False
        t = 0
        state = env.reset()
        state = np.reshape(state, [1, state_size])
        while not done:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            next_state = np.reshape(next_state, [1, state_size])
            reward = reward if not done else -100
            agent.add_experience(state, action, reward, next_state, done)
            agent.experience_replay()
            t += 1
            state = next_state
            if done:
                agent.update_target_model()
                score.append(t)
                print(f"Episode: {e}, score: {t}, epsilon: {agent.epsilon:.3f}")
                break
        if np.mean(score[-min(100, len(score)):]) >= (env.spec.max_episode_steps - 5):
            print(f'Problem is solved in {e} episodes.')
            break
    env.close()
    return score

In [7]:
if __name__ == '__main__':
    visualize_cartpole(episodes=3)


/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/core.py:49: DeprecationWarning: WARN: You are calling render method, but you didn't specified the argument render_mode at environment initialization. To mainta

Episode 0 done after 21 steps
Episode 1 done after 25 steps
Episode 2 done after 57 steps


In [8]:
if __name__ == '__main__':
    q_learning_cartpole(max_episodes=200)


/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [15]:
if __name__ == '__main__':
    run_dqn_cartpole(episodes=50)

/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/core.py:256: DeprecationWarning: WARN: Function `env.seed(seed)` is marked as deprecated and will be removed in the future. Please use `env.reset(seed=seed)` i

Episode: 0, score: 16, epsilon: 1.000
Episode: 1, score: 28, epsilon: 1.000
Episode: 2, score: 13, epsilon: 1.000
Episode: 3, score: 26, epsilon: 1.000
Episode: 4, score: 30, epsilon: 1.000
Episode: 5, score: 31, epsilon: 1.000
Episode: 6, score: 26, epsilon: 1.000
Episode: 7, score: 75, epsilon: 1.000
Episode: 8, score: 29, epsilon: 1.000
Episode: 9, score: 25, epsilon: 1.000
Episode: 10, score: 13, epsilon: 1.000
Episode: 11, score: 14, epsilon: 1.000
Episode: 12, score: 21, epsilon: 1.000
Episode: 13, score: 21, epsilon: 1.000
Episode: 14, score: 25, epsilon: 1.000
Episode: 15, score: 17, epsilon: 1.000
Episode: 16, score: 15, epsilon: 1.000
Episode: 17, score: 32, epsilon: 1.000
Episode: 18, score: 19, epsilon: 1.000
Episode: 19, score: 29, epsilon: 1.000
Episode: 20, score: 14, epsilon: 1.000
Episode: 21, score: 23, epsilon: 1.000
Episode: 22, score: 25, epsilon: 1.000
Episode: 23, score: 17, epsilon: 1.000
Episode: 24, score: 19, epsilon: 1.000
Episode: 25, score: 26, epsilon: 1.

KeyboardInterrupt: 